# 800 nm Interlocking: Relative Phase Between Vortex Components

Loads the calibrated Xe "interlocking error bars" experiment and extracts the **relative phase**
$\Delta\eta(E)$ between the two counter-rotating vortex components as a function of photoelectron
energy.

Reference: `800 Interlocking Error Bars.ipynb`.

## Model

The two vortex arms carry opposite OAM ($\ell=\pm1$), so their coherent sum in the
$(p_r,\phi)$ plane gives

$$ I(p_r,\phi) \propto A_+^2 + A_-^2 + 2 A_+ A_- \cos\!\big(2\phi + \Delta\eta(E)\big), $$

with $\Delta\eta(E) = \eta_+(E) - \eta_-(E) = \arg c_6 - \arg c_4$ (bright arm $m=6$ minus
weak arm $m=4$). The interference minima therefore sit at
$2\,\phi_{\min}(E) + \Delta\eta(E) = \pi \pmod{2\pi}$, i.e.

$$ \Delta\eta(E) = \pi - 2\,\phi_{\min}(E) \pmod{2\pi}. $$

`extract_theory_df` returns the smoothed fringe angle $\theta(E)\equiv\phi_{\min}(E)$; the
relative phase is $\Delta\eta = \pi - 2\theta$ (up to a constant convention offset set by
`ANGULAR_SHIFT`) &mdash; the **same** direction as the model box above and the
`arctan2(-b2, a2)` modulation phase used by the reconstruction cells below.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy
import skimage
from matplotlib import colors
from scipy.ndimage import rotate
from scipy.signal import argrelmin
from sklearn.cluster import DBSCAN

# atomic units -> physical units
ENERGY_CONV = 27.2114        # 1 a.u. energy = 27.2114 eV
TIME_CONV_FS = 0.02418884    # 1 a.u. time   = 0.02418884 fs


def bump(x):
    return np.where(np.abs(x) < 1, np.exp(1 / (x ** 2 - 1) + 1) ** 0.2, 0)


def argrelmin_rotated(arr, angle, axis=0, order=1, reshape=True, interp_order=1,
                      mode='constant', cval=np.inf):
    """Relative minima of a 2D array along lines at `angle` deg, mapped back to original coords."""
    arr = np.asarray(arr)
    M, N = arr.shape
    arr_rot = rotate(arr, -angle, reshape=reshape, order=interp_order, mode=mode, cval=cval)
    M_rot, N_rot = arr_rot.shape
    rows_rot, cols_rot = argrelmin(arr_rot, axis=axis, order=order)
    center_orig = np.array([(M - 1) / 2.0, (N - 1) / 2.0])
    center_rot = np.array([(M_rot - 1) / 2.0, (N_rot - 1) / 2.0])
    theta = np.deg2rad(angle)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    dr = rows_rot - center_rot[0]
    dc = cols_rot - center_rot[1]
    col_orig = cos_t * dc + sin_t * dr + center_orig[1]
    row_orig = -sin_t * dc + cos_t * dr + center_orig[0]
    row_idx = np.round(row_orig).astype(int)
    col_idx = np.round(col_orig).astype(int)
    valid = (row_idx >= 0) & (row_idx < M) & (col_idx >= 0) & (col_idx < N)
    return row_idx[valid], col_idx[valid]


def extract_theory_df(I, r_v, theta_v, sigma_small=25, sigma_large=150, angle=60,
                      poly_order=12, spline_lambda=1e-7, e_scale=1):
    """Return DataFrame ['pr','E','theta','group_delay','signal'] tracing the interference fringe.

    `theta` is the smoothed fringe angle phi_min(E); the vortex relative phase is
    Delta eta = pi - 2*theta = arg c6 - arg c4 (up to a constant convention offset),
    and `group_delay` = d(Delta eta)/dE = -2 * d(theta)/dE.
    """
    fft = np.fft.fft(I, axis=1)
    tot = np.abs(fft[:, 0])
    recovered = skimage.filters.difference_of_gaussians(I, sigma_small, sigma_large, mode='wrap')

    args_r = argrelmin_rotated(recovered, angle=angle, axis=1, order=1)
    r_vals, theta_vals = r_v[args_r[0]], theta_v[args_r[1]]

    mask = (r_vals > 0.13) & (r_vals < 0.3) & (theta_vals > 0) & (theta_vals < 2)
    r_min, theta_min = r_vals[mask], theta_vals[mask]

    labels = DBSCAN(eps=0.1, min_samples=1).fit_predict(np.vstack((r_min * 15, theta_min)).T)
    sel = labels == np.argmax(np.bincount(labels))
    r_min, theta_min = r_min[sel], theta_min[sel]

    df_pts = pd.DataFrame({'r': r_min, 'theta': theta_min})
    df_pts = df_pts.groupby('theta', as_index=False)['r'].min()
    r_min, theta_min = df_pts['r'].values, df_pts['theta'].values

    crs = np.max(I, axis=1) - np.min(I, axis=1)
    idxs = np.intersect1d(theta_v, theta_min, return_indices=True)[1]
    cr = crs[idxs]

    E_min = r_min ** 2 / 2
    coeffs = np.polyfit(E_min, theta_min, poly_order, w=cr)
    r_sm = np.linspace(r_min.min(), r_min.max(), len(r_min))
    E = r_sm ** 2 / 2
    theta_sm = np.poly1d(coeffs)(E)
    group_delay = np.gradient(theta_sm, E * e_scale) * -2   # d(Delta eta)/dE, Delta eta = pi - 2*theta

    tot_sm = scipy.interpolate.make_smoothing_spline(r_v ** 2 / 2, tot, lam=spline_lambda)(E)

    return pd.DataFrame({
        'pr': r_min,
        'E': E * e_scale,
        'theta': theta_sm,
        'group_delay': group_delay,
        'signal': tot_sm / np.max(tot_sm),
    })

## Load experimental data (calibrated Xe)

In [ ]:
fname_exp = r"J:\ctgroup\Edward\DATA\VMI\20220613\xe005_e_c.h5"
base_data = pd.read_hdf(fname_exp, key="data")

exp = (base_data
       .query("sqrt(px**2+py**2+pz**2)<0.4")
       .query("abs(py)<0.5")
       .query("pz>0")
       .reset_index(drop=True))
exp_sym = exp.copy()
exp_sym[['px', 'py', 'pz']] *= -1
exp = pd.concat([exp, exp_sym], ignore_index=True)

exp['pr'] = np.linalg.norm(exp[['px', 'py', 'pz']].values, axis=1)
exp['phi'] = np.arctan2(exp['px'], exp['pz'])
exp['E'] = exp['pr'] ** 2 / 2

RES = 1024
h2d, re_edges, pe_edges = np.histogram2d(exp['pr'], exp['phi'], bins=RES)
r_vals = 0.5 * (re_edges[:-1] + re_edges[1:])
theta_vals = 0.5 * (pe_edges[:-1] + pe_edges[1:])
I_exp = (h2d.T * bump((r_vals - 0.25) / 0.1)).T
print('experiment grid:', I_exp.shape)

## Diagnostics: fringe-contrast budget

Before extracting phases, check where the fringe **contrast** is lost. Contrast here is the
peak-to-trough $(\max-\min)/(\max+\min)$ of the binned angular distribution $I(\phi)$
(`NB_C` bins) at the radial-profile peak ring, with an out-of-plane acceptance
$|p_y| < 0.05$ a.u. (`PY_ACC`):

* **contrast vs number of $\phi$ bins** &mdash; max&ndash;min inflates with bin count under
  shot noise; the fundamental $2\phi$ projection amplitude is drawn alongside for reference;
* **contrast vs radial-shell half-width** &mdash; a thick shell mixes fringe angles and
  dilutes contrast (the pipeline uses one ~$4\times10^{-4}$ a.u. bin, marked);
* **raw per-radial-bin projection** $v(E)$ vs the **`make_smoothing_spline`** preview at
  several `lam` &mdash; over-smoothing $I_{\rm mod}$ relative to $I_0$ (as in cell
  `a9f34885`) is the usual contrast killer (this block keeps the full $|p_y|<0.5$ histogram
  `h2d`, since it probes the spline, not the $p_y$ cut);
* the **angular power spectrum** $|C_m|/|C_0|$ at the ring &mdash; pedestal ($m=0$), the
  $2\phi$ fringe, and leakage to other $m$;
* **contrast vs the $|p_y|$ acceptance** cut (out-of-plane events sit off the bright ring;
  the adopted $0.05$ cut is marked).

In [ ]:
import scipy.interpolate
from plotly.subplots import make_subplots

# ============================================================================
# DIAGNOSTICS -- fringe-contrast budget
# contrast  C = (max - min) / (max + min)  of the binned angular distribution
# I(phi) at the reference ring, with an out-of-plane acceptance |p_y| < PY_ACC.
# Trace C through phi-binning / radial-shell width / smoothing-spline /
# harmonic leakage / p_y acceptance, before any phase extraction, to see
# where the reconstruction loses contrast relative to the raw events.
# ============================================================================
EXP_SCALE_DIAG = 0.874 ** 2                       # == exp_scale in the ensemble cell
PY_ACC = 0.05                                     # out-of-plane acceptance: |p_y| < PY_ACC
NB_C = 16                                         # phi bins for the max-min contrast

_n0 = len(exp) // 2                               # exp = [ measured ; sign-flipped copy ]
_d = exp.iloc[:_n0]                               # independent events only
_ph = _d['phi'].to_numpy()                        # in (-pi/2, pi/2): one full 2-phi period
_pr = _d['pr'].to_numpy()
_py = _d['py'].to_numpy()
_RNG = (-np.pi / 2, np.pi / 2)
_acc = np.abs(_py) < PY_ACC                       # p_y acceptance mask


def _contrast(sel, nbins=NB_C):
    """peak-to-trough contrast  (max - min) / (max + min)  of the phi-histogram."""
    hh, _ = np.histogram(_ph[sel], bins=nbins, range=_RNG)
    if hh.sum() < 200 or hh.max() + hh.min() == 0:
        return np.nan
    return (hh.max() - hh.min()) / (hh.max() + hh.min())


def _vis_proj(sel, nbins=NB_C):
    """fundamental 2-phi amplitude: project the phi-histogram onto cos/sin(2 phi)."""
    hh, ee = np.histogram(_ph[sel], bins=nbins, range=_RNG)
    ct = 0.5 * (ee[:-1] + ee[1:])
    i0 = hh.mean()
    if i0 <= 0:
        return np.nan
    a = 2 * (hh * np.cos(2 * ct)).mean()
    b = 2 * (hh * np.sin(2 * ct)).mean()
    return np.hypot(a, b) / i0


# reference ring: radial-profile peak within the fringe band
_bpr = (r_vals > 0.13) & (r_vals < 0.32)
_rpk = r_vals[_bpr][np.argmax(h2d.sum(axis=1)[_bpr])]
_Epk = _rpk ** 2 / 2 * EXP_SCALE_DIAG * ENERGY_CONV
_sh0 = (np.abs(_pr - _rpk) < 0.015) & _acc        # reference shell, p_y-accepted

_nref = int(np.count_nonzero(_sh0))
print(f'reference ring : pr = {_rpk:.3f} a.u.   E = {_Epk:.3f} eV   '
      f'n = {_nref}   (|p_y| < {PY_ACC})')
print(f'  contrast (max-min)/(max+min), {NB_C} bins : {_contrast(_sh0):.3f}   <-- reported contrast')
print(f'  2-phi projection amplitude               : {_vis_proj(_sh0):.3f}')
print(f'  same, no p_y cut                         : '
      f'C = {_contrast(np.abs(_pr - _rpk) < 0.015):.3f}   '
      f'proj = {_vis_proj(np.abs(_pr - _rpk) < 0.015):.3f}')

# per-radial-bin projection exactly as cell a9f34885 (raw), + smoothing-spline preview
# (h2d keeps the full |p_y| < 0.5 acceptance -- this block probes the spline, not p_y)
_cc, _sc = np.cos(2 * theta_vals), np.sin(2 * theta_vals)
I0_bin = h2d.mean(axis=1)
Imod_bin = np.hypot(2 * (h2d * _cc).mean(axis=1), 2 * (h2d * _sc).mean(axis=1))
E_bin = r_vals ** 2 / 2 * EXP_SCALE_DIAG
_m = _bpr & (I0_bin > 0)
_x = E_bin[_m]
_vraw = Imod_bin[_m] / I0_bin[_m]
print(f'\nband: {_m.sum()} radial bins   Imod>I0 (would be clipped): '
      f'{int(np.count_nonzero(Imod_bin[_m] > I0_bin[_m]))}')
print(f'  raw per-bin projection    mean {_vraw.mean():.3f}   median {np.median(_vraw):.3f}')
_spl = {}
for _lam in (1e-10, 1e-9, 1e-8):
    _i0s = scipy.interpolate.make_smoothing_spline(_x, I0_bin[_m], lam=_lam)(_x)
    _ims = scipy.interpolate.make_smoothing_spline(_x, Imod_bin[_m], lam=_lam)(_x)
    _spl[_lam] = np.clip(_ims, 0, _i0s) / _i0s
    print(f'  spline lam={_lam:.0e}      mean projection {_spl[_lam].mean():.3f}')

# averaged angular profile at the ring (full 2 pi) -> harmonic content
_rowsel = np.abs(r_vals - _rpk) < 0.010
_prof = h2d[_rowsel].sum(axis=0).astype(float)
_Cm = np.abs(np.fft.rfft(_prof)) / _prof.sum()
print(f'\nangular harmonics |C_m|/|C_0| at ring : '
      + '  '.join(f'm{mm}={_Cm[mm]:.3f}' for mm in (1, 2, 3, 4, 6)))
print(f'\n==> reported contrast {_contrast(_sh0):.3f}   2-phi projection {_vis_proj(_sh0):.3f}   '
      f'raw per-bin mean {_vraw.mean():.3f}')

# ----------------------------------------------------------------- figure ------
_nb = np.array([8, 12, 16, 24, 32, 48, 64, 96, 128])
_c_nb = np.array([_contrast(_sh0, int(n)) for n in _nb])
_p_nb = np.array([_vis_proj(_sh0, int(n)) for n in _nb])
_hw = np.array([0.004, 0.006, 0.008, 0.010, 0.013, 0.017, 0.022, 0.030, 0.040, 0.055])
_c_hw = np.array([_contrast((np.abs(_pr - _rpk) < h) & _acc) for h in _hw])
_pyc = np.array([0.03, 0.05, 0.08, 0.12, 0.16, 0.20, 0.28, 0.40, 0.50])
_c_py = np.array([_contrast((np.abs(_pr - _rpk) < 0.015) & (np.abs(_py) < p)) for p in _pyc])

_nbp = 48
_pb, _ee = np.histogram(_ph[_sh0], bins=_nbp, range=_RNG)
_ct = 0.5 * (_ee[:-1] + _ee[1:])
_pb = _pb / _pb.mean()
_af = 2 * (_pb * np.cos(2 * _ct)).mean()
_bf = 2 * (_pb * np.sin(2 * _ct)).mean()
_fit = 1 + _af * np.cos(2 * _ct) + _bf * np.sin(2 * _ct)

ddg = make_subplots(rows=3, cols=2, vertical_spacing=0.10, horizontal_spacing=0.11,
                    subplot_titles=(
                        "contrast vs phi-bin count (reference ring)",
                        "contrast vs radial-shell half-width",
                        "per-bin projection v(E): raw vs smoothing-spline",
                        "angular harmonics |C_m| / |C_0|",
                        "angular profile at ring + 2-phi fit",
                        "contrast vs |p_y| acceptance"))

ddg.add_scatter(x=_nb, y=_c_nb, mode='markers+lines', line_color='royalblue',
                name='max-min', showlegend=True, row=1, col=1)
ddg.add_scatter(x=_nb, y=_p_nb, mode='markers+lines', line_color='gray',
                name='2-phi proj', showlegend=True, row=1, col=1)
ddg.add_vline(x=NB_C, line=dict(color='crimson', dash='dash'),
              annotation_text=f'{NB_C} bins', row=1, col=1)
ddg.update_xaxes(type='log', title_text='phi bins', row=1, col=1)

ddg.add_scatter(x=_hw, y=_c_hw, mode='markers+lines', line_color='seagreen', row=1, col=2)
ddg.add_vline(x=r_vals[1] - r_vals[0], line=dict(color='gray', dash='dot'),
              annotation_text='pipeline bin', row=1, col=2)
ddg.update_xaxes(title_text='shell half-width (a.u.)', row=1, col=2)

ddg.add_scatter(x=_x * ENERGY_CONV, y=_vraw, mode='markers',
                marker=dict(size=3, color='lightslategray'), row=2, col=1)
for _lam, _col in zip((1e-10, 1e-9, 1e-8), ('#1f77b4', '#ff7f0e', '#d62728')):
    ddg.add_scatter(x=_x * ENERGY_CONV, y=_spl[_lam], mode='lines', line_color=_col,
                    name=f'lam={_lam:.0e}', row=2, col=1)
ddg.update_xaxes(title_text='Energy (eV)', range=[0.28, 0.82], row=2, col=1)

ddg.add_bar(x=np.arange(9), y=_Cm[:9], marker_color='slateblue', row=2, col=2)
ddg.update_xaxes(title_text='harmonic m', row=2, col=2)

ddg.add_bar(x=_ct, y=_pb, marker_color='lightgray', row=3, col=1)
ddg.add_scatter(x=_ct, y=_fit, mode='lines', line_color='crimson', row=3, col=1)
ddg.update_xaxes(title_text='phi (rad)', row=3, col=1)

ddg.add_scatter(x=_pyc, y=_c_py, mode='markers+lines', line_color='purple', row=3, col=2)
ddg.add_vline(x=PY_ACC, line=dict(color='crimson', dash='dash'),
              annotation_text=f'|p_y| < {PY_ACC}', row=3, col=2)
ddg.update_xaxes(title_text='|p_y| cut (a.u.)', row=3, col=2)

ddg.update_yaxes(rangemode='tozero')
ddg.update_layout(template='simple_white+presentation', width=1000, height=1050,
                  showlegend=False, title=f'Contrast diagnostics (ring E = {_Epk:.2f} eV)')
ddg.show()


## Extract the fringe angle for an ensemble of smoothing parameters

Each choice of difference-of-Gaussians scales / probe angle gives one estimate of $\theta(E)$.
The spread across the ensemble is the systematic uncertainty band.

In [ ]:
def phase_ensemble(I_grid, r_grid, theta_grid, e_scale=1.0,
                   angles=None, highs=None, lows=None):
    """Run extract_theory_df over a sweep of angle / sigma_large / sigma_small."""
    angles = np.arange(0, 90, 3) if angles is None else angles
    highs = np.arange(120, 164, 4) if highs is None else highs
    lows = np.arange(12, 40, 2) if lows is None else lows
    dfs = []
    for angle in angles:
        try:
            dfs.append(extract_theory_df(I_grid, r_grid, theta_grid, angle=angle, e_scale=e_scale))
        except Exception as err:
            print(f'angle={angle}: {err}')
    for hs in highs:
        try:
            dfs.append(extract_theory_df(I_grid, r_grid, theta_grid, sigma_large=hs, e_scale=e_scale))
        except Exception as err:
            print(f'hs={hs}: {err}')
    for ls in lows:
        try:
            dfs.append(extract_theory_df(I_grid, r_grid, theta_grid, sigma_small=ls, e_scale=e_scale))
        except Exception as err:
            print(f'ls={ls}: {err}')
    return dfs


exp_scale = 0.874 ** 2   # r-axis calibration factor from the reference notebook

exp_dfs = phase_ensemble(I_exp, r_vals, theta_vals, e_scale=exp_scale)
print(len(exp_dfs), 'experiment curves')

## Resample onto a common energy grid and reduce to relative phase

`rel_phase = pi - 2*theta` (radians) $=\Delta\eta=\arg c_6-\arg c_4$ &mdash; the same
direction as the reconstruction. `ANGULAR_SHIFT` sets the absolute-phase convention (the
vortex relative phase is only defined up to an overall constant).

In [ ]:
def reduce_phase(dfs, E_grid, angular_shift=0.0):
    th = np.vstack([np.interp(E_grid, df['E'], df['theta']) for df in dfs])
    gd = np.vstack([np.interp(E_grid, df['E'], df['group_delay']) for df in dfs])
    # relative phase in the model direction: Delta eta = pi - 2*phi_min = arg c6 - arg c4
    rel = np.pi - 2.0 * th + 2.0 * angular_shift
    return {
        'phase_mean': rel.mean(axis=0), 'phase_std': rel.std(axis=0),
        'gd_mean': gd.mean(axis=0), 'gd_std': gd.std(axis=0),
    }


NUM_POINTS = 100
ANGULAR_SHIFT = 0.6035 - np.radians(10)   # absolute-phase convention (reference notebook)

all_E = np.hstack([df['E'].values for df in exp_dfs])
E_grid = np.linspace(all_E.min(), all_E.max(), NUM_POINTS)
E_ev = E_grid * ENERGY_CONV

# analysis / display energy window, widened 50% about its centre (0.40-0.70 -> 0.325-0.775 eV)
E_BAND = (0.4, 0.7)

experi = reduce_phase(exp_dfs, E_grid, angular_shift=ANGULAR_SHIFT)

phase_df = pd.DataFrame({
    'E_au': E_grid,
    'E_eV': E_ev,
    'exp_phase_rad': experi['phase_mean'],
    'exp_phase_std': experi['phase_std'],
})
phase_df.head()

In [ ]:
def add_banded_line(fig, x, y, dy, color, name, showlegend=True):
    fig.add_scatter(x=x, y=y + dy, line=dict(width=0, color=color), showlegend=False, hoverinfo='skip')
    fig.add_scatter(x=x, y=y - dy, line=dict(width=0, color=color), showlegend=False, hoverinfo='skip',
                    fill='tonexty', fillcolor=f"rgba{colors.to_rgba(color, alpha=0.3)}")
    fig.add_scatter(x=x, y=y, mode='lines', line=dict(width=2, color=color), name=name, showlegend=showlegend)


fig = go.Figure().update_layout(
    template='simple_white+presentation',
    width=650, height=450,
    title="Vortex Relative Phase vs Energy (Xe experiment)",
    xaxis_title="Energy (eV)",
    yaxis_title="Relative Phase Δη (rad)",
    xaxis=dict(range=list(E_BAND), ticks='inside', mirror=True),
    yaxis=dict(range=[1, 3.5], ticks='inside', mirror=True),   # re-tune with ANGULAR_SHIFT after the sign flip
    legend=dict(orientation='h', yanchor='top', y=1, xanchor='right', x=1),
)
add_banded_line(fig, E_ev, experi['phase_mean'], experi['phase_std'], 'red', 'Experiment')
fig.show()

## Vortex-state intensities from the modulation strength

For each radial bin take the binned angular profile $I(\phi)$ (coarse `NB_C` $\phi$-bins,
$|p_y| < 0.05$ a.u. acceptance `PY_ACC`) and smooth each $\phi$-bin vs energy. The
modulation is the **peak-to-trough** $I_{\rm mod} = \tfrac12(\max-\min)$ of the smoothed
profile &mdash; smoothing *before* $\max-\min$ keeps Poisson fluctuations from biasing the
contrast high &mdash; with pedestal $I_0 = \langle I\rangle_\phi$.

With $I_0 = A_+^2 + A_-^2$ and $I_{\rm mod} = 2 A_+ A_-$ the two vortex intensities are the
roots of $x^2 - I_0\,x + (I_{\rm mod}/2)^2 = 0$:

$$ A_\pm^2 = \tfrac{1}{2}\Big( I_0 \pm \sqrt{I_0^2 - I_{\rm mod}^2}\,\Big). $$

The modulation *phase* $\Delta\eta$ is still taken from the $2\phi$ cos/sin projection
(`arctan2(-b2, a2)` in the next cell), since $\max-\min$ carries no phase. Symmetrization of
the data makes the odd harmonics vanish, so $m=2$ is the leading modulation term.

In [ ]:
# --- angular decomposition per radial bin, with |p_y| < PY_ACC acceptance -------
# (PY_ACC, NB_C are defined in the diagnostics cell 2be611e5)
_acc2 = exp['py'].abs().to_numpy() < PY_ACC
h2d_py, _, _ = np.histogram2d(exp['pr'].to_numpy()[_acc2], exp['phi'].to_numpy()[_acc2],
                              bins=[re_edges, pe_edges])

c2, s2 = np.cos(2 * theta_vals), np.sin(2 * theta_vals)
I0_r = h2d_py.mean(axis=1)
a2 = 2 * (h2d_py * c2).mean(axis=1)                       # 2*phi projection -> modulation phase
b2 = 2 * (h2d_py * s2).mean(axis=1)                       #   (used for Delta eta in cell 703c9053)

E_r = (r_vals ** 2 / 2) * exp_scale
band = (r_vals > 0.13) & (r_vals < 0.32) & (I0_r > 0)

# coarse angular profile (NB_C phi bins), in counts-per-fine-bin units
NB_INT = NB_C
_fold = h2d_py.shape[1] // NB_INT
prof_c = (h2d_py[:, :NB_INT * _fold]
          .reshape(h2d_py.shape[0], NB_INT, _fold).sum(axis=2) / _fold)   # (r, NB_C)
Imod_r = 0.5 * (prof_c.max(axis=1) - prof_c.min(axis=1))  # per-bin (max - min)/2  (diagnostic)

# smooth each angular bin vs energy, THEN take peak-to-trough -- smoothing before
# max-min keeps Poisson fluctuations from biasing the contrast high
lam = 1e-8
prof_sm = np.vstack([
    scipy.interpolate.make_smoothing_spline(E_r[band], prof_c[band, k], lam=lam)(E_grid)
    for k in range(NB_INT)
]).T                                                      # (E_grid, NB_C)
I0_sm = prof_sm.mean(axis=1)
Imod_sm = 0.5 * (prof_sm.max(axis=1) - prof_sm.min(axis=1))
Imod_sm = np.clip(Imod_sm, 0, I0_sm)                      # visibility <= 1

root = np.sqrt(np.clip(I0_sm ** 2 - Imod_sm ** 2, 0, None))
A2_strong = (I0_sm + root) / 2                            # brighter vortex component
A2_weak = (I0_sm - root) / 2                              # dimmer vortex component
visibility = Imod_sm / I0_sm                              # peak-to-trough contrast, (max-min)/(2 mean)

phase_df['I0'] = I0_sm
phase_df['modulation'] = Imod_sm
phase_df['visibility'] = visibility
phase_df['vortex_strong'] = A2_strong
phase_df['vortex_weak'] = A2_weak

vfig = go.Figure().update_layout(
    template='simple_white+presentation', width=650, height=450,
    title=f"Vortex-state intensities vs energy  (|p_y| < {PY_ACC}, max-min modulation)",
    xaxis_title="Energy (eV)", yaxis_title="Intensity (counts)",
    xaxis=dict(range=list(E_BAND), ticks='inside', mirror=True),
    yaxis=dict(ticks='inside', mirror=True),
    legend=dict(orientation='h', yanchor='top', y=1, xanchor='right', x=1),
)
vfig.add_scatter(x=E_ev, y=A2_strong, mode='lines', name='Strong vortex', line_color='crimson')
vfig.add_scatter(x=E_ev, y=A2_weak, mode='lines', name='Weak vortex', line_color='royalblue')
vfig.add_scatter(x=E_ev, y=I0_sm, mode='lines', name='Total', line=dict(color='black', dash='dot'))
vfig.show()


## Complex field of the two vortex components  (m = 6 and m = 4)

Assign the OAM quantum numbers: the bright component is $m_1 = 6$, the weak one $m_2 = 4$,
so the beat order is $\Delta m = m_1 - m_2 = 2$ &mdash; consistent with the $2\phi$ modulation
fitted above.

Write the angular field as

$$ E(\phi, E) = c_6(E)\,e^{i 6\phi} + c_4(E)\,e^{i 4\phi}, \qquad
   c_6 = A_6\,e^{i\,\Delta\eta(E)}, \quad c_4 = A_4 , $$

with $A_6 = \sqrt{A_+^2}$, $A_4 = \sqrt{A_-^2}$ from the modulation decomposition and
$\Delta\eta(E)$ the per-ring fitted modulation phase (`arctan2(-b2, a2)`, unwrapped and
smoothed). The overall phase is fixed by $\arg c_4 \equiv 0$; $\Delta\eta$ is defined only
mod $2\pi$.

In [ ]:
M1, M2 = 6, 4                       # OAM of bright / weak vortex component
assert abs(M1 - M2) == 2, "beat order must match the fitted 2*phi modulation"

# fitted modulation phase per radial bin -> Delta eta(E)
delta_r = np.arctan2(-b2, a2)                       # Delta eta per ring (wrapped)
Es = E_r[band]
order = np.argsort(Es)
delta_unwrap = np.unwrap(delta_r[band][order])
delta_eta = scipy.interpolate.make_smoothing_spline(
    Es[order], delta_unwrap, lam=1e-8)(E_grid)

# modal amplitudes from the intensity decomposition
A6 = np.sqrt(np.clip(A2_strong, 0, None))
A4 = np.sqrt(np.clip(A2_weak, 0, None))

# complex modal coefficients (global phase: arg c4 = 0)
c6 = A6 * np.exp(1j * delta_eta)
c4 = A4.astype(complex)

# reconstructed complex angular field E(phi, E)
phi_recon = np.linspace(-np.pi, np.pi, 512, endpoint=False)
field = (c6[:, None] * np.exp(1j * M1 * phi_recon)[None, :]
         + c4[:, None] * np.exp(1j * M2 * phi_recon)[None, :])

phase_df['delta_eta_rad'] = delta_eta
phase_df['A6'] = A6
phase_df['A4'] = A4
phase_df['c6_re'], phase_df['c6_im'] = c6.real, c6.imag
phase_df['c4_re'], phase_df['c4_im'] = c4.real, c4.imag

jpk = int(np.argmax(I0_sm))
print(f'peak E = {E_ev[jpk]:.3f} eV:  c6 = {c6[jpk]:.2f}  (|c6|={A6[jpk]:.2f}), '
      f'c4 = {c4[jpk]:.2f}  (|c4|={A4[jpk]:.2f}),  '
      f'Delta eta = {np.degrees(delta_eta[jpk]) % 360:.1f} deg')

sel = (E_ev > E_BAND[0]) & (E_ev < E_BAND[1])
ffig = go.Figure().update_layout(
    template='simple_white+presentation', width=650, height=550,
    title=f"Complex modal coefficients c_m(E)  (Argand, E = {E_BAND[0]:.3g}-{E_BAND[1]:.3g} eV)",
    xaxis_title="Re", yaxis_title="Im",
    xaxis=dict(ticks='inside', mirror=True, scaleanchor='y', scaleratio=1),
    yaxis=dict(ticks='inside', mirror=True),
    legend=dict(orientation='h', yanchor='top', y=1, xanchor='right', x=1),
)
ffig.add_scatter(x=c6[sel].real, y=c6[sel].imag, mode='markers+lines',
                 marker=dict(size=5, color=E_ev[sel], colorscale='Viridis',
                             colorbar=dict(title='E (eV)')),
                 line=dict(color='crimson', width=1), name='c6 (m=6)')
ffig.add_scatter(x=c4[sel].real, y=c4[sel].imag, mode='markers+lines',
                 marker=dict(size=5, color='royalblue'),
                 line=dict(color='royalblue', width=1), name='c4 (m=4)')
ffig.add_scatter(x=[0], y=[0], mode='markers', marker=dict(size=8, color='black', symbol='x'),
                 showlegend=False)
ffig.show()

# reconstructed |E|^2 ring pattern vs the measured intensity
recon_2d = np.abs(field) ** 2
rfig = go.Figure().update_layout(
    template='simple_white+presentation', width=680, height=420,
    title="Reconstructed |E(phi, E)|^2  from c6 e^{i6phi} + c4 e^{i4phi}",
    xaxis_title="phi (rad)", yaxis_title="Energy (eV)",
)
rfig.add_heatmap(x=phi_recon, y=E_ev, z=recon_2d, colorscale='Inferno', colorbar=dict(title='|E|^2'))
rfig.update_yaxes(range=list(E_BAND))
rfig.show()

### Reconstruction vs. measurement, and the position-space distribution

First figure &mdash; left: the measured $I(\phi, E)$ (raw histogram, resampled onto the
reconstruction grid); right: the reconstructed $|E(\phi,E)|^2$ from
$c_6 e^{i6\phi} + c_4 e^{i4\phi}$. Both maps are normalised to their own mean over the
`E_BAND` window, so the shared colour scale reads as fractional modulation $1\pm v$.

Second figure ($2\times3$) &mdash; three momentum-space fields (top) and their **2D Fourier
transforms** to position space (bottom). Common ingredients: modal magnitudes
($A_6$, $A_4$ via $I_0$, $I_{\rm mod}$) and relative phase $\Delta\eta$ re-evaluated over the
full fitted band `E_r[band]` and then **Gaussian-smoothed** (`RECON_SMOOTH`, in `E_recon`
points); phase convention $\arg c_6 - \arg c_4 = +\Delta\eta$ ($c_6$ real,
$c_4 = A_4 e^{-i\Delta\eta}$); smooth `bump` radial gate; `PAD_FACTOR` zero-padding before
`fft2`.

* **reconstruction** &mdash; the analytic field $c_6 e^{i6\phi} + c_4 e^{i4\phi}$ with the
  fitted $A_6, A_4$ (top-middle shows $|E(p_x,p_z)|^2$).
* **hybrid** &mdash; amplitude $\sqrt{I_{\rm meas}(p_x,p_z)}$ from the measured histogram
  ($|p_y|<0.05$, momenta $\times\sqrt{\texttt{exp\_scale}}$, `MEAS_SMOOTH`-px blur); phase
  $\arg E$ from the reconstruction (top-right, cyclic colour).
* **forced magnitude** &mdash; the analytic field with the two OAM arms rescaled to a common
  envelope `A_eq` (optionally a prefactor on the weak arm) to probe how the vortex-amplitude
  ratio drives the position-space contrast.

Bottom row: $|\psi(x,z)|^2$ for each field, **each normalised to its own max** (shape
comparison), cropped to where they carry power. The $\Delta m = 2$ beat becomes a two-lobe
structure in position; a more balanced amplitude ratio deepens the null between the lobes.

In [ ]:
from plotly.subplots import make_subplots

# --- measured I(phi, E): resample the raw histogram onto the reconstruction grid ---
order_r = np.argsort(E_r)
_meas_E = np.vstack([np.interp(E_grid, E_r[order_r], h2d[order_r, j])
                     for j in range(h2d.shape[1])]).T          # (E_grid, theta_vals)
meas_2d = np.vstack([np.interp(phi_recon, theta_vals, _meas_E[i], period=2 * np.pi)
                     for i in range(E_grid.size)])             # (E_grid, phi_recon)

selE = (E_ev > E_BAND[0]) & (E_ev < E_BAND[1])
# normalise each map to its mean over the band so the two colour scales show the
# same thing -- fractional modulation 1 +/- v -- not a single (noisy) peak pixel
meas_n = meas_2d / np.nanmean(meas_2d[selE])
recon_n = recon_2d / np.nanmean(recon_2d[selE])
_cmax = float(np.nanpercentile(np.r_[meas_n[selE].ravel(), recon_n[selE].ravel()], 99))

cfig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.07,
                     subplot_titles=("Measured  I(phi, E)",
                                     "Reconstructed  |E(phi, E)|^2"))
cfig.add_heatmap(x=phi_recon, y=E_ev, z=meas_n, colorscale='Inferno', zmin=0, zmax=_cmax,
                 colorbar=dict(title='I / mean'), row=1, col=1)
cfig.add_heatmap(x=phi_recon, y=E_ev, z=recon_n, colorscale='Inferno', zmin=0, zmax=_cmax,
                 showscale=False, row=1, col=2)
cfig.update_xaxes(title_text="phi (rad)")
cfig.update_yaxes(range=list(E_BAND))
cfig.update_yaxes(title_text="Energy (eV)", row=1, col=1)
cfig.update_layout(template='simple_white+presentation', width=920, height=430,
                   title="Reconstruction vs. measurement")
cfig.show()

# report the fringe contrast of each map over the band (row-wise min/max)
_mn, _mx = meas_n[selE].min(axis=1), meas_n[selE].max(axis=1)
_rn, _rx = recon_n[selE].min(axis=1), recon_n[selE].max(axis=1)
print(f'band contrast (max-min)/(max+min):  measured {np.mean((_mx - _mn) / (_mx + _mn)):.3f}   '
      f'reconstruction {np.mean((_rx - _rn) / (_rx + _rn)):.3f}')

# --- Cartesian (px, pz) reconstruction: modal splines over the full fitted band ---
E_band = E_r[band]                                   # strictly increasing (r_vals ascending)
E_recon = np.linspace(E_band.min(), E_band.max(), 240)
pr_grid = np.sqrt(2 * E_recon)                       # calibrated |p| over the full band

RECON_SMOOTH = 18                                    # gaussian sigma (E_recon pts) -- significant smoothing

_prof_w = np.vstack([
    scipy.interpolate.make_smoothing_spline(E_band, prof_c[band, k], lam=lam)(E_recon)
    for k in range(NB_INT)
]).T
# true peak-to-trough of the measured angular profile: Fourier-interpolate the
# NB_C-wedge profile to a fine phi grid and take its actual min/max -- recovers the
# non-sinusoidal sharpness the discrete wedges clip
_prof_fine = scipy.signal.resample(_prof_w, 512, axis=1)
_I0_w = _prof_fine.mean(axis=1)
_Imod_w = np.clip(0.5 * (_prof_fine.max(axis=1) - _prof_fine.min(axis=1)), 0, _I0_w)

# relative phase: the same smoothing spline as cell 703c9053 -- arctan2(-b2, a2),
# unwrapped, lam=1e-8 -- evaluated on E_recon
deta_w = scipy.interpolate.make_smoothing_spline(
    E_band, np.unwrap(delta_r[band]), lam=1e-8)(E_recon)

# significant smoothing of the modal magnitudes (via I0, Imod) and the relative phase
_I0_w = scipy.ndimage.gaussian_filter1d(_I0_w, RECON_SMOOTH, mode='nearest')
_Imod_w = np.clip(scipy.ndimage.gaussian_filter1d(_Imod_w, RECON_SMOOTH, mode='nearest'), 0, _I0_w)
deta_w = scipy.ndimage.gaussian_filter1d(deta_w, RECON_SMOOTH, mode='nearest')

_root_w = np.sqrt(np.clip(_I0_w ** 2 - _Imod_w ** 2, 0, None))
A6_w = np.sqrt(np.clip((_I0_w + _root_w) / 2, 0, None))
A4_w = np.sqrt(np.clip((_I0_w - _root_w) / 2, 0, None))

NXY = 400
P_MAX = 0.30
pax = np.linspace(-P_MAX, P_MAX, NXY)
dp = pax[1] - pax[0]
PX, PZ = np.meshgrid(pax, pax)
PR = np.hypot(PX, PZ)
PHI = np.arctan2(PX, PZ)                            # same convention as exp['phi']


def radial_interp(cvals):
    re = np.interp(PR, pr_grid, cvals.real, left=0.0, right=0.0)
    im = np.interp(PR, pr_grid, cvals.imag, left=0.0, right=0.0)
    return re + 1j * im


# reconstructed complex field: c6 real, c4 = A4 exp(-i Delta eta)  -> arg c6 - arg c4 = +Delta eta
c6_flat = A6_w.astype(complex)
c4_phase = A4_w * np.exp(-1j * deta_w)
field_core = (radial_interp(c6_flat) * np.exp(1j * M1 * PHI)
              + 1.6*radial_interp(c4_phase) * np.exp(1j * M2 * PHI))
# field_core=radial_interp(c6_flat)+radial_interp(c4_phase)*np.exp(1j*2*PHI)
# equal-magnitude reconstruction: |c6| = |c4| = sqrt(I0 / 2)  -> full-contrast fringes,
# same radial envelope and same Delta eta(E) as field_core
A_eq = np.sqrt(np.clip(_I0_w / 2, 0, None))
field_eq = (radial_interp(A_eq.astype(complex)) * np.exp(1j * M1 * PHI)
            +0.5* radial_interp(A_eq * np.exp(-1j * deta_w)) * np.exp(1j * M2 * PHI))

# soft radial gate (bump) spanning the annulus -> no hard edges for the FFT
pr_c = 0.5 * (pr_grid.min() + pr_grid.max())
pr_hw = 0.5 * (pr_grid.max() - pr_grid.min())
gate = bump((PR - pr_c) / pr_hw)

# --- measured amplitude on this grid (for the hybrid field) -------------------
MEAS_SMOOTH = 0                                    # gaussian sigma (px) on the measured map
_cal = np.sqrt(exp_scale)                            # raw momenta -> calibrated
_selpy = exp['py'].abs().to_numpy() < PY_ACC
_edg = np.linspace(pax[0] - dp / 2, pax[-1] + dp / 2, NXY + 1)
I_meas_xy = np.histogram2d(exp['px'].to_numpy()[_selpy] * _cal,
                           exp['pz'].to_numpy()[_selpy] * _cal, bins=[_edg, _edg])[0].T
I_meas_xy = scipy.ndimage.gaussian_filter(I_meas_xy, MEAS_SMOOTH)

phase_rec = np.angle(field_core)
recon_I = np.abs(field_core) ** 2                    # reconstructed momentum intensity (ungated)

# --- zero-padded 2D Fourier transforms -> position-space distributions ---------
PAD_FACTOR = 12
NPAD = NXY * PAD_FACTOR
pad = (NPAD - NXY) // 2
x_ax = np.fft.fftshift(np.fft.fftfreq(NPAD, d=dp)) * (2 * np.pi)    # conjugate position (a.u.)


def fft_pos(f):
    fp = np.pad(f * gate, pad)                                     # gate + zero pad
    psi = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(fp))) * (dp ** 2 / (2 * np.pi))
    return np.abs(psi) ** 2


pos_rec = fft_pos(field_core)                                      # analytic reconstruction
field_hyb = np.sqrt(np.clip(I_meas_xy, 0, None)) * np.exp(1j * phase_rec)
pos_hyb = fft_pos(field_hyb)                                       # measured |E|, reconstructed phase
pos_eq = fft_pos(field_eq)                                         # forced-magnitude reconstruction


def _crop(*maps):
    mx = np.maximum.reduce([m / m.max() for m in maps])
    xm = np.where(mx.max(axis=0) > 1e-3)[0]
    zm = np.where(mx.max(axis=1) > 1e-3)[0]
    return slice(xm[0], xm[-1] + 1), slice(zm[0], zm[-1] + 1)


xs, zs = _crop(pos_rec, pos_hyb, pos_eq)
_pr_n = pos_rec / pos_rec[zs, xs].max()                            # each position map -> own max
_ph_n = pos_hyb / pos_hyb[zs, xs].max()
_pe_n = pos_eq / pos_eq[zs, xs].max()
# clip each position plot below 0.5 * its max
_pr_n = np.where(_pr_n >= 0.45, _pr_n, 0.0)
_ph_n = np.where(_ph_n >= 0.5, _ph_n, 0.0)
_pe_n = np.where(_pe_n >= 0.5, _pe_n, 0.0)

# --- figure: momentum-space (top) and position-space FTs (bottom) -----------
_ph_show = np.where(gate > 0.4, phase_rec, np.nan)

pfig = make_subplots(rows=2, cols=3, vertical_spacing=0.14, horizontal_spacing=0.10,
                     subplot_titles=(
                         f"Measured  I(p_x, p_z)   (|p_y| < {PY_ACC})",
                         "Reconstructed  |E(p_x, p_z)|^2",
                         "Reconstruction phase  arg E",
                         "|psi(x, z)|^2   reconstruction  (clipped < 0.5 max)",
                         "|psi(x, z)|^2   hybrid  (clipped < 0.5 max)",
                         "|psi(x, z)|^2   forced magnitude  (clipped < 0.5 max)"))

pfig.add_heatmap(x=pax, y=pax, z=I_meas_xy, colorscale='Inferno',
                 colorbar=dict(title='counts', x=0.295, y=0.79, len=0.42, thickness=10), row=1, col=1)
pfig.add_heatmap(x=pax, y=pax, z=recon_I, colorscale='Inferno',
                 colorbar=dict(title='|E|^2', x=0.655, y=0.79, len=0.42, thickness=10), row=1, col=2)
pfig.add_heatmap(x=pax, y=pax, z=_ph_show, colorscale='HSV', zmin=-np.pi, zmax=np.pi,
                 colorbar=dict(title='arg E', x=1.0, y=0.79, len=0.42, thickness=10), row=1, col=3)
pfig.add_heatmap(x=x_ax[xs], y=x_ax[zs], z=_pr_n[zs, xs], colorscale='Inferno',
                 zmin=0, zmax=1, showscale=False, row=2, col=1)
pfig.add_heatmap(x=x_ax[xs], y=x_ax[zs], z=_ph_n[zs, xs], colorscale='Inferno',
                 zmin=0, zmax=1, showscale=False, row=2, col=2)
pfig.add_heatmap(x=x_ax[xs], y=x_ax[zs], z=_pe_n[zs, xs], colorscale='Inferno',
                 zmin=0, zmax=1,
                 colorbar=dict(title='|psi|^2 (norm.)', x=1.0, y=0.21, len=0.42, thickness=10),
                 row=2, col=3)

pfig.update_xaxes(title_text="p_x (a.u.)", row=1, col=1)
pfig.update_yaxes(title_text="p_z (a.u.)", scaleanchor='x', scaleratio=1, row=1, col=1)
pfig.update_xaxes(title_text="p_x (a.u.)", row=1, col=2)
pfig.update_yaxes(scaleanchor='x2', scaleratio=1, row=1, col=2)
pfig.update_xaxes(title_text="p_x (a.u.)", row=1, col=3)
pfig.update_yaxes(scaleanchor='x3', scaleratio=1, row=1, col=3)
pfig.update_xaxes(title_text="x (a.u.)", row=2, col=1)
pfig.update_yaxes(title_text="z (a.u.)", scaleanchor='x4', scaleratio=1, row=2, col=1)
pfig.update_xaxes(title_text="x (a.u.)", row=2, col=2)
pfig.update_yaxes(scaleanchor='x5', scaleratio=1, row=2, col=2)
pfig.update_xaxes(title_text="x (a.u.)", row=2, col=3)
pfig.update_yaxes(scaleanchor='x6', scaleratio=1, row=2, col=3)
pfig.update_layout(template='simple_white+presentation', width=1300, height=820,
                   title="Momentum reconstructions and their position-space 2D Fourier transforms")
pfig.show()

# --- forced-magnitude field: coherent sum intensity and phase ---------------
eq_I = np.abs(field_eq) ** 2                                       # |c6 e^{i6phi} + c4 e^{i4phi}|^2
eq_ph = np.where(gate > 0.4, np.angle(field_eq), np.nan)
efig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                     subplot_titles=("Forced magnitude  |E_eq(p_x, p_z)|^2   (sum)",
                                     "Forced magnitude phase  arg E_eq"))
efig.add_heatmap(x=pax, y=pax, z=eq_I, colorscale='Inferno',
                 colorbar=dict(title='|E_eq|^2', x=0.42, thickness=12), row=1, col=1)
efig.add_heatmap(x=pax, y=pax, z=eq_ph, colorscale='HSV', zmin=-np.pi, zmax=np.pi,
                 colorbar=dict(title='arg E_eq', x=1.0, thickness=12), row=1, col=2)
efig.update_xaxes(title_text="p_x (a.u.)", row=1, col=1)
efig.update_yaxes(title_text="p_z (a.u.)", scaleanchor='x', scaleratio=1, row=1, col=1)
efig.update_xaxes(title_text="p_x (a.u.)", row=1, col=2)
efig.update_yaxes(title_text="p_z (a.u.)", scaleanchor='x2', scaleratio=1, row=1, col=2)
efig.update_layout(template='simple_white+presentation', width=900, height=430,
                   title="Forced-magnitude reconstruction (|c6| = |c4|): coherent sum intensity and phase")
efig.show()


## Save extracted relative phase

In [ ]:
out_csv = r"C:\Users\mcman\DataspellProjects\vmi-analysis\800_interlocking_relative_phase.csv"
phase_df.to_csv(out_csv, index=False)
print('wrote', out_csv)